# Nmap LLM Fine-tuning on Google Colab

Fine-tune Qwen model to execute complete nmap scanning workflows.

## Steps

1. **Runtime** → Change runtime type → GPU (T4 recommended)
2. Run cells in order
3. Get HuggingFace token from https://huggingface.co/settings/tokens

## What This Teaches

- Complete scan workflows (discovery → ports → services → vulns)
- Target types (IP, CIDR, domain)
- Phase transitions
- Error handling & fallbacks

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers datasets accelerate peft bitsandbytes huggingface_hub

In [ ]:
# Step 2: Login to HuggingFace
from huggingface_hub import login
login()

In [ ]:
# Step 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted!")

In [ ]:
# Step 4: Clone your repo
# REPLACE 'YOUR_USERNAME' and 'YOUR_REPO' with your GitHub info
!git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/nmap-llama
print("Repo cloned!")

In [ ]:
# Step 5: Generate training data
import sys
sys.path.insert(0, '/content/nmap-llama/finetune')

# Import the generator
exec(open('/content/nmap-llama/finetune/generate_training_data.py').read())

# Run it
main()

In [ ]:
# Step 6: Import libraries
import os
import json
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 7: Configuration
# Change MODEL_NAME to Qwen/Qwen2-1.5B or Qwen/Qwen2-7B for better results

MODEL_NAME = "Qwen/Qwen2-0.5B"  # Base model
OUTPUT_DIR = "/content/drive/MyDrive/nmap-finetuned"  # Where to save

BATCH_SIZE = 2
LEARNING_RATE = 2e-4
NUM_EPOCHS = 5
MAX_LENGTH = 768

In [ ]:
# Step 8: Load training data
DATA_PATH = "/content/nmap-llama/finetune/data/training_qwen.txt"

texts = []
with open(DATA_PATH, 'r') as f:
    for line in f:
        texts.append(line.strip())

print(f"Loaded {len(texts)} training examples")

# Create dataset
dataset = Dataset.from_dict({'text': texts})
print(f"Dataset size: {len(dataset)}")

In [ ]:
# Step 9: Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded!")

In [ ]:
# Step 10: Load model with 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)
print("Model loaded!")

In [ ]:
# Step 11: Configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 12: Tokenize data
def tokenize(examples):
    result = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    result['labels'] = result['input_ids'].copy()
    return result

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=['text'])
print(f"Tokenized: {len(tokenized_dataset)} examples")

In [ ]:
# Step 13: Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
    warmup_steps=20,
    optim="paged_adamw_8bit"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Ready to train!")

In [ ]:
# Step 14: TRAIN! (This takes 15-30 min with T4)
print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Step 15: Save model
os.makedirs(f"{OUTPUT_DIR}/final", exist_ok=True)
trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"Model saved to {OUTPUT_DIR}/final")

In [ ]:
# Step 16: Test the model
from transformers import pipeline

# Load fine-tuned model
test_model = AutoModelForCausalLM.from_pretrained(
    f"{OUTPUT_DIR}/final",
    device_map="auto",
    trust_remote_code=True
)
test_tokenizer = AutoTokenizer.from_pretrained(f"{OUTPUT_DIR}/final")

pipe = pipeline("text-generation", model=test_model, tokenizer=test_tokenizer)

# Test prompts
prompts = [
    "Target: 192.168.1.0/24\nType: CIDR Range\nGoal: Full network scan",
    "Target: 10.0.0.5\nType: Single IP\nGoal: Full security assessment"
]

for prompt in prompts:
    print(f"\n{'='*60}")
    print(f"PROMPT:\n{prompt}")
    result = pipe(prompt, max_new_tokens=300, do_sample=False)
    print(f"\nRESPONSE:\n{result[0]['generated_text']}")

---

## Using with Ollama

```bash
# Save model to HuggingFace
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path=f"{OUTPUT_DIR}/final",
    repo_id="YOUR_USERNAME/nmap-qwen-finetuned",
    repo_type="model"
)

# Or convert for Ollama:
# 1. Go to https://huggingface.co/settings/tokens
# 2. Create new token with 'Write' access
# 3. Use ollama create or llama.cpp to convert
```